# Experiment 02 — Eye Swin V2 + LBP + GLCM + Gabor + Wavelet Fusion

**Purpose:** Binary real/fake classification from `combined` eye ROI images.

This Colab notebook follows the uploaded ML project standard:

- fixed seed and reproducible run IDs
- source-video-level split leakage checks
- configuration-driven execution
- train-only scaler fitting
- atomic feature/cache/checkpoint writes
- `last.ckpt` and `best.ckpt`
- smoke test before full training
- English plots with minimum 600 px short edge
- prediction, metrics, ablation and audit outputs

> The notebook does not alter raw ROI files. It reads the existing eye ROI metadata and `combined_eye_path` values.

In [1]:
# ============================================================
# 0. COLAB SETUP
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!pip -q install \
    "torch>=2.3" "torchvision>=0.18" \
    "scikit-image>=0.24" "PyWavelets>=1.6" \
    "scikit-learn>=1.4" "pandas>=2.0" "numpy>=1.26" \
    "matplotlib>=3.8" "Pillow>=10.0" "PyYAML>=6.0" \
    "tqdm>=4.66" "joblib>=1.3"

Mounted at /content/drive


In [7]:
from pathlib import Path

for p in Path("/content/drive/MyDrive").rglob("metadata.csv"):
    print(p)

/content/drive/MyDrive/metadata.csv


In [27]:
# ============================================================
# 1. CONFIGURATION
# Edit only this block before the first run.
# ============================================================
from pathlib import Path

CONFIG = {
    # Input metadata produced by the eye ROI extraction pipeline.
    "metadata_csv": (
        "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
        "Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv"
    ),

    # Experiment outputs. Raw data is never modified.
    "output_root": (
        "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
        "Deneyler/Kader/Sonuçlar/SwinV2_TextureFusion"
    ),

    # Metadata field names.
    "path_column": "combined_eye_path",
    "label_column": "label",
    "split_column": "split",
    "video_column": "video_id",
    "sample_id_column": "sample_id",
    "status_column": "status",

    # Accepted successful states in the existing ROI report.
    "success_status_values": ["ok", "success", "SUCCESS"],

    # Image/model settings.
    "image_size": 224,
    "model_name": "swin_v2_t",
    "pretrained": True,
    "freeze_backbone_epochs": 5,
    "unfreeze_last_stages": True,

    # Handcrafted texture settings.
    "lbp_radii": [1, 2, 3],
    "lbp_points": [8, 16, 24],
    "glcm_distances": [1, 2, 4],
    "glcm_angles_deg": [0, 45, 90, 135],
    "glcm_levels": 32,
    "gabor_orientations_deg": [0, 30, 60, 90, 120, 150],
    "gabor_frequencies": [0.10, 0.20, 0.30],
    "wavelet": "db2",
    "wavelet_level": 2,

    # Training.
    "seed": 42,
    "batch_size": 16,
    "num_workers": 2,
    "epochs": 30,
    "learning_rate": 2e-4,
    "backbone_learning_rate": 2e-5,
    "weight_decay": 1e-4,
    "dropout": 0.30,
    "hidden_dim": 256,
    "patience": 8,
    "threshold": 0.50,
    "threshold_search_min": 0.10,
    "threshold_search_max": 0.90,
    "threshold_search_steps": 161,
    "amp": True,
    "gradient_clip_norm": 1.0,

    # Development controls.
    "smoke_test": True,
    "smoke_train_batches": 2,
    "smoke_val_batches": 2,
    "feature_cache_flush_every": 100,
    "overwrite_feature_cache": False,

    # Ablation modes. Full mode is the main experiment.
    "ablation_modes": [
        "swin_only",
        "texture_only",
        "swin_lbp",
        "swin_lbp_glcm",
        "swin_lbp_glcm_gabor",
        "full",
    ],
    "run_all_ablations": False,
    "active_mode": "full",
}

assert CONFIG["active_mode"] in CONFIG["ablation_modes"]
print("Configuration loaded.")

Configuration loaded.


In [30]:
# ============================================================
# 2. IMPORTS, SEED, RUN ID AND OUTPUT DIRECTORIES
# ============================================================
import os
import gc
import io
import math
import time
import copy
import json
import yaml
import random
import hashlib
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import cv2
import joblib
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from skimage.feature import (
    local_binary_pattern,
    graycomatrix,
    graycoprops,
)
from skimage.filters import gabor

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import swin_v2_t, Swin_V2_T_Weights
from torchvision import transforms

warnings.filterwarnings("ignore", category=UserWarning)

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ID = datetime.now().strftime(
    f"%Y%m%d_%H%M_eye_swinv2_texturefusion_seed{CONFIG['seed']}"
)

RUN_DIR = Path(CONFIG["output_root"]) / RUN_ID
DIRS = {
    "run": RUN_DIR,
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "features": RUN_DIR / "features",
    "audit": RUN_DIR / "audit",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

with open(RUN_DIR / "config_resolved.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

print("Device:", DEVICE)
print("Run ID:", RUN_ID)
print("Run directory:", RUN_DIR)

Device: cuda
Run ID: 20260806_1748_eye_swinv2_texturefusion_seed42
Run directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Sonuçlar/SwinV2_TextureFusion/20260806_1748_eye_swinv2_texturefusion_seed42


In [31]:
# ============================================================
# 3. ATOMIC I/O HELPERS
# ============================================================
def atomic_write_bytes(data: bytes, target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    with open(temp, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp, target)

def atomic_write_text(text: str, target: Path) -> None:
    atomic_write_bytes(text.encode("utf-8"), target)

def atomic_write_json(payload: Dict[str, Any], target: Path) -> None:
    atomic_write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str),
        target,
    )

def atomic_write_csv(df: pd.DataFrame, target: Path) -> None:
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    atomic_write_text(buffer.getvalue(), target)

def atomic_joblib_dump(obj: Any, target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    joblib.dump(obj, temp)
    _ = joblib.load(temp)
    os.replace(temp, target)

def atomic_torch_save(state: Dict[str, Any], target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    torch.save(state, temp)
    loaded = torch.load(temp, map_location="cpu", weights_only=False)
    required = {"epoch", "model_state_dict", "optimizer_state_dict"}
    if not required.issubset(loaded):
        raise RuntimeError(f"Checkpoint validation failed: {temp}")
    os.replace(temp, target)

print("Atomic I/O helpers ready.")

Atomic I/O helpers ready.


In [32]:
# ============================================================
# 4. METADATA LOAD, NORMALIZATION AND QUALITY GATES
# ============================================================
from pathlib import Path
import pandas as pd

metadata_path = Path(CONFIG["metadata_csv"])

if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata bulunamadı: {metadata_path}")

metadata = pd.read_csv(metadata_path)

print("Toplam metadata satırı:", len(metadata))
print("Sütunlar:", metadata.columns.tolist())

# Metinleri düzenle
metadata[CONFIG["label_column"]] = (
    metadata[CONFIG["label_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

metadata[CONFIG["split_column"]] = (
    metadata[CONFIG["split_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "training": "train",
        "validation": "val",
        "valid": "val",
        "testing": "test",
    })
)

metadata[CONFIG["status_column"]] = (
    metadata[CONFIG["status_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("\nStatus değerleri:")
print(metadata[CONFIG["status_column"]].value_counts(dropna=False).head(20))

print("\nLabel değerleri:")
print(metadata[CONFIG["label_column"]].value_counts(dropna=False))

print("\nSplit değerleri:")
print(metadata[CONFIG["split_column"]].value_counts(dropna=False))

# combined_eye_path dolu olan ve doğru label/split içeren satırları al
valid = metadata[
    metadata[CONFIG["path_column"]].notna()
    & metadata[CONFIG["label_column"]].isin(["real", "fake"])
    & metadata[CONFIG["split_column"]].isin(["train", "val", "test"])
].copy()

valid["target"] = (
    valid[CONFIG["label_column"]]
    .map({"real": 0, "fake": 1})
    .astype(int)
)

ROI_ROOT = Path(
    "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
    "Deneyler/Kader/Deney 1/Göz/eye_roi_output"
)

valid["resolved_image_path"] = valid[
    CONFIG["path_column"]
].astype(str).map(
    lambda p: str(ROI_ROOT / p)
)

# Gerçekten diskte olan görselleri tut
valid["file_exists"] = valid["resolved_image_path"].map(
    lambda p: Path(p).exists()
)

print("\nFiltre sonrası satır:", len(valid))
print("Diskte bulunan görsel:", valid["file_exists"].sum())

missing_files = valid[~valid["file_exists"]].copy()
valid = valid[valid["file_exists"]].copy()

if valid.empty:
    print("\nİlk 10 combined_eye_path:")
    print(
        metadata[CONFIG["path_column"]]
        .dropna()
        .astype(str)
        .head(10)
        .to_string(index=False)
    )

    raise RuntimeError(
        "Combined göz yolları CSV'de var ama dosyalar diskte bulunamadı."
    )

if not valid[CONFIG["sample_id_column"]].is_unique:
    valid = valid.drop_duplicates(
        subset=[CONFIG["sample_id_column"]],
        keep="first"
    )

# Video sızıntısı kontrolü
video_sets = {
    split: set(
        valid.loc[
            valid[CONFIG["split_column"]] == split,
            CONFIG["video_column"],
        ].astype(str)
    )
    for split in ["train", "val", "test"]
}

assert video_sets["train"].isdisjoint(video_sets["val"])
assert video_sets["train"].isdisjoint(video_sets["test"])
assert video_sets["val"].isdisjoint(video_sets["test"])

print("\nBaşarılı.")
print("Kullanılacak toplam örnek:", len(valid))
print(
    valid.groupby(
        [CONFIG["split_column"], CONFIG["label_column"]]
    ).size()
)

Toplam metadata satırı: 3097
Sütunlar: ['sample_id', 'source_frame', 'relative_frame_path', 'label', 'split', 'video_id', 'frame_stem', 'face_id', 'face_id_scope', 'faces_in_frame', 'left_eye_detected', 'right_eye_detected', 'left_eye_path', 'right_eye_path', 'combined_eye_path', 'debug_path', 'landmarks_path', 'left_bbox_x1', 'left_bbox_y1', 'left_bbox_x2', 'left_bbox_y2', 'right_bbox_x1', 'right_bbox_y1', 'right_bbox_x2', 'right_bbox_y2', 'combined_bbox_x1', 'combined_bbox_y1', 'combined_bbox_x2', 'combined_bbox_y2', 'left_eye_width_px', 'left_eye_height_px', 'right_eye_width_px', 'right_eye_height_px', 'left_iris_landmarks_available', 'right_iris_landmarks_available', 'left_iris_visible_estimate', 'right_iris_visible_estimate', 'landmark_count', 'status', 'error', 'processing_ms']

Status değerleri:
status
ok         2986
no_face     111
Name: count, dtype: int64

Label değerleri:
label
fake    1566
real    1531
Name: count, dtype: int64

Split değerleri:
split
train    2478
test   

In [33]:
# ============================================================
# 5. DATA DISTRIBUTION FIGURES
# ============================================================
def save_figure(fig: plt.Figure, name: str) -> Path:
    target = DIRS["figures"] / name
    fig.savefig(target, dpi=150, bbox_inches="tight")
    plt.close(fig)

    with Image.open(target) as image:
        if min(image.size) < 600:
            raise AssertionError(
                f"Figure resolution is below the minimum: {image.size}"
            )
    return target

split_class = (
    valid.groupby([CONFIG["split_column"], CONFIG["label_column"]])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
split_class.plot(kind="bar", ax=ax)
ax.set_title("Eye ROI Class Distribution by Dataset Split")
ax.set_xlabel("Dataset Split")
ax.set_ylabel("Number of Samples")
ax.legend(title="Class")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, "01_dataset_distribution.png")

print("Dataset distribution figure saved.")

Dataset distribution figure saved.


In [34]:
# ============================================================
# 6. HANDCRAFTED FEATURE EXTRACTION
# LBP + GLCM + Gabor + Wavelet
# ============================================================
def read_gray_image(path: str, size: int = 224) -> np.ndarray:
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Image could not be read: {path}")
    if image.shape != (size, size):
        image = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
    return image

def safe_entropy(values: np.ndarray, bins: int = 64) -> float:
    flat = np.asarray(values, dtype=np.float64).ravel()
    if flat.size == 0:
        return 0.0
    hist, _ = np.histogram(flat, bins=bins, density=False)
    probs = hist.astype(np.float64)
    total = probs.sum()
    if total <= 0:
        return 0.0
    probs /= total
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())

def extract_lbp(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    features = []
    names = []
    for radius, points in zip(CONFIG["lbp_radii"], CONFIG["lbp_points"]):
        lbp = local_binary_pattern(
            gray,
            P=points,
            R=radius,
            method="uniform",
        )
        bins = points + 2
        hist, _ = np.histogram(
            lbp.ravel(),
            bins=np.arange(0, bins + 1),
            range=(0, bins),
        )
        hist = hist.astype(np.float32)
        hist /= hist.sum() + 1e-8
        features.extend(hist.tolist())
        names.extend([f"lbp_r{radius}_p{points}_bin{i}" for i in range(bins)])
    return np.asarray(features, dtype=np.float32), names

def extract_glcm(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    levels = int(CONFIG["glcm_levels"])
    quantized = np.floor(gray.astype(np.float32) / 256.0 * levels)
    quantized = np.clip(quantized, 0, levels - 1).astype(np.uint8)

    angles = np.deg2rad(CONFIG["glcm_angles_deg"])
    matrix = graycomatrix(
        quantized,
        distances=CONFIG["glcm_distances"],
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True,
    )

    properties = [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM",
    ]
    features = []
    names = []
    for prop in properties:
        values = graycoprops(matrix, prop)
        for d_idx, distance in enumerate(CONFIG["glcm_distances"]):
            for a_idx, angle in enumerate(CONFIG["glcm_angles_deg"]):
                features.append(float(values[d_idx, a_idx]))
                names.append(f"glcm_{prop}_d{distance}_a{angle}")
    return np.asarray(features, dtype=np.float32), names

def extract_gabor(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    image = gray.astype(np.float32) / 255.0
    features = []
    names = []

    for frequency in CONFIG["gabor_frequencies"]:
        for angle_deg in CONFIG["gabor_orientations_deg"]:
            theta = np.deg2rad(angle_deg)
            real, imag = gabor(image, frequency=frequency, theta=theta)
            magnitude = np.sqrt(real ** 2 + imag ** 2)

            stats = {
                "mean": float(magnitude.mean()),
                "std": float(magnitude.std()),
                "energy": float(np.mean(magnitude ** 2)),
                "entropy": safe_entropy(magnitude),
            }
            for stat_name, value in stats.items():
                features.append(value)
                names.append(
                    f"gabor_f{frequency:.2f}_a{angle_deg}_{stat_name}"
                )

    return np.asarray(features, dtype=np.float32), names

def coefficient_statistics(
    coefficients: np.ndarray,
    prefix: str,
) -> Tuple[List[float], List[str]]:
    array = np.asarray(coefficients, dtype=np.float64)
    abs_array = np.abs(array)
    stats = {
        "mean": float(array.mean()),
        "std": float(array.std()),
        "energy": float(np.mean(array ** 2)),
        "abs_mean": float(abs_array.mean()),
        "abs_max": float(abs_array.max()),
        "entropy": safe_entropy(array),
    }
    return list(stats.values()), [f"{prefix}_{key}" for key in stats]

def extract_wavelet(gray: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    image = gray.astype(np.float32) / 255.0
    coefficients = pywt.wavedec2(
        image,
        wavelet=CONFIG["wavelet"],
        level=CONFIG["wavelet_level"],
        mode="symmetric",
    )

    features = []
    names = []

    approximation = coefficients[0]
    values, value_names = coefficient_statistics(
        approximation,
        f"wavelet_L{CONFIG['wavelet_level']}_LL",
    )
    features.extend(values)
    names.extend(value_names)

    detail_levels = coefficients[1:]
    current_level = CONFIG["wavelet_level"]
    for horizontal, vertical, diagonal in detail_levels:
        for band_name, band in [
            ("LH", horizontal),
            ("HL", vertical),
            ("HH", diagonal),
        ]:
            values, value_names = coefficient_statistics(
                band,
                f"wavelet_L{current_level}_{band_name}",
            )
            features.extend(values)
            names.extend(value_names)
        current_level -= 1

    return np.asarray(features, dtype=np.float32), names

def extract_texture_features(path: str) -> Tuple[np.ndarray, Dict[str, slice], List[str]]:
    gray = read_gray_image(path, CONFIG["image_size"])

    groups = {}
    all_features = []
    all_names = []
    start = 0

    for group_name, extractor in [
        ("lbp", extract_lbp),
        ("glcm", extract_glcm),
        ("gabor", extract_gabor),
        ("wavelet", extract_wavelet),
    ]:
        values, names = extractor(gray)
        end = start + len(values)
        groups[group_name] = slice(start, end)
        start = end
        all_features.append(values)
        all_names.extend(names)

    vector = np.concatenate(all_features).astype(np.float32)
    if not np.isfinite(vector).all():
        raise FloatingPointError(f"NaN/Inf texture feature detected: {path}")

    return vector, groups, all_names

# Feature dimension smoke check.
example_vector, FEATURE_SLICES, FEATURE_NAMES = extract_texture_features(
    valid.iloc[0]["resolved_image_path"]
)
print("Texture feature dimension:", len(example_vector))
print("Feature slices:", FEATURE_SLICES)

Texture feature dimension: 240
Feature slices: {'lbp': slice(0, 54, None), 'glcm': slice(54, 126, None), 'gabor': slice(126, 198, None), 'wavelet': slice(198, 240, None)}


In [35]:
# ============================================================
# 7. BUILD OR RESUME ATOMIC TEXTURE FEATURE CACHE
# ============================================================
FEATURE_CACHE_ROOT = Path(CONFIG["output_root"]) / "_feature_cache"
FEATURE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

cache_key_payload = {
    "image_size": CONFIG["image_size"],
    "lbp_radii": CONFIG["lbp_radii"],
    "lbp_points": CONFIG["lbp_points"],
    "glcm_distances": CONFIG["glcm_distances"],
    "glcm_angles_deg": CONFIG["glcm_angles_deg"],
    "glcm_levels": CONFIG["glcm_levels"],
    "gabor_orientations_deg": CONFIG["gabor_orientations_deg"],
    "gabor_frequencies": CONFIG["gabor_frequencies"],
    "wavelet": CONFIG["wavelet"],
    "wavelet_level": CONFIG["wavelet_level"],
}
CACHE_VERSION = hashlib.sha256(
    json.dumps(cache_key_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

CACHE_FILE = FEATURE_CACHE_ROOT / f"texture_features_{CACHE_VERSION}.joblib"
CACHE_MANIFEST = FEATURE_CACHE_ROOT / f"texture_manifest_{CACHE_VERSION}.csv"

if CACHE_FILE.exists() and not CONFIG["overwrite_feature_cache"]:
    texture_cache = joblib.load(CACHE_FILE)
else:
    texture_cache = {}

pending_rows = valid[
    ~valid[CONFIG["sample_id_column"]].astype(str).isin(texture_cache.keys())
]

print(f"Existing cached samples: {len(texture_cache):,}")
print(f"Pending samples: {len(pending_rows):,}")

cache_errors = []
processed_since_flush = 0

for _, row in tqdm(
    pending_rows.iterrows(),
    total=len(pending_rows),
    desc="Extracting texture features",
):
    sample_id = str(row[CONFIG["sample_id_column"]])
    image_path = row["resolved_image_path"]

    try:
        vector, _, _ = extract_texture_features(image_path)
        texture_cache[sample_id] = vector
    except Exception as exc:
        cache_errors.append(
            {
                "sample_id": sample_id,
                "image_path": image_path,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
        )

    processed_since_flush += 1
    if processed_since_flush >= CONFIG["feature_cache_flush_every"]:
        atomic_joblib_dump(texture_cache, CACHE_FILE)
        processed_since_flush = 0

atomic_joblib_dump(texture_cache, CACHE_FILE)

if cache_errors:
    atomic_write_csv(
        pd.DataFrame(cache_errors),
        DIRS["audit"] / "texture_feature_errors.csv",
    )

valid["texture_cached"] = valid[CONFIG["sample_id_column"]].astype(str).isin(
    texture_cache.keys()
)
feature_manifest = valid.loc[valid["texture_cached"]].copy()
atomic_write_csv(feature_manifest, CACHE_MANIFEST)
atomic_write_text("\n".join(FEATURE_NAMES), FEATURE_CACHE_ROOT / f"feature_names_{CACHE_VERSION}.txt")

valid = feature_manifest.reset_index(drop=True)
if valid.empty:
    raise RuntimeError("Texture feature extraction produced no valid sample.")


print(f"Final cached samples used: {len(valid):,}")

Existing cached samples: 2,986
Pending samples: 0


Extracting texture features: 0it [00:00, ?it/s]

Final cached samples used: 2,986


In [18]:
# ============================================================
# 8. TRAIN-ONLY TEXTURE SCALER
# ============================================================
train_ids = valid.loc[
    valid[CONFIG["split_column"]] == "train",
    CONFIG["sample_id_column"],
].astype(str)

train_texture = np.stack([texture_cache[sid] for sid in train_ids])
texture_scaler = StandardScaler()
texture_scaler.fit(train_texture)

SCALER_FILE = DIRS["features"] / "texture_scaler.joblib"
atomic_joblib_dump(texture_scaler, SCALER_FILE)

atomic_write_json(
    {
        "fit_split": "train",
        "feature_dimension": int(train_texture.shape[1]),
        "cache_version": CACHE_VERSION,
        "feature_groups": {
            name: [group_slice.start, group_slice.stop]
            for name, group_slice in FEATURE_SLICES.items()
        },
    },
    DIRS["features"] / "texture_feature_schema.json",
)

print("Texture scaler fitted only on the training split.")

Texture scaler fitted only on the training split.


In [36]:
# ============================================================
# 9. DATASET AND AUGMENTATION
# ============================================================

# Swin V2-Tiny ImageNet normalization values
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Texture feature'lar orijinal görüntüden çıkarıldığı için
# geometrik augmentation uygulanmıyor.
train_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["image_size"], CONFIG["image_size"])
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["image_size"], CONFIG["image_size"])
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])

MODE_GROUPS = {
    "swin_only": [],
    "texture_only": [
        "lbp",
        "glcm",
        "gabor",
        "wavelet",
    ],
    "swin_lbp": [
        "lbp",
    ],
    "swin_lbp_glcm": [
        "lbp",
        "glcm",
    ],
    "swin_lbp_glcm_gabor": [
        "lbp",
        "glcm",
        "gabor",
    ],
    "full": [
        "lbp",
        "glcm",
        "gabor",
        "wavelet",
    ],
}


def select_texture_groups(
    vector: np.ndarray,
    mode: str,
) -> np.ndarray:
    groups = MODE_GROUPS[mode]

    if not groups:
        return np.zeros((0,), dtype=np.float32)

    selected = [
        vector[FEATURE_SLICES[group]]
        for group in groups
    ]

    return np.concatenate(selected).astype(np.float32)


class EyeFusionDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        mode: str,
        transform,
        scaler: StandardScaler,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.mode = mode
        self.transform = transform
        self.scaler = scaler

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(
        self,
        index: int,
    ) -> Dict[str, Any]:

        row = self.df.iloc[index]

        image_path = row["resolved_image_path"]

        sample_id = str(
            row[CONFIG["sample_id_column"]]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        image_tensor = self.transform(image)

        raw_texture = texture_cache[
            sample_id
        ].reshape(1, -1)

        scaled_texture = self.scaler.transform(
            raw_texture
        )[0].astype(np.float32)

        selected_texture = select_texture_groups(
            scaled_texture,
            self.mode,
        )

        return {
            "image": image_tensor,
            "texture": torch.from_numpy(
                selected_texture
            ),
            "target": torch.tensor(
                int(row["target"]),
                dtype=torch.float32,
            ),
            "sample_id": sample_id,
            "video_id": str(
                row[CONFIG["video_column"]]
            ),
            "image_path": image_path,
        }


def build_loaders(mode: str):

    split_frames = {
        split: valid[
            valid[CONFIG["split_column"]] == split
        ].copy()
        for split in [
            "train",
            "val",
            "test",
        ]
    }

    datasets = {
        "train": EyeFusionDataset(
            split_frames["train"],
            mode,
            train_transform,
            texture_scaler,
        ),
        "val": EyeFusionDataset(
            split_frames["val"],
            mode,
            eval_transform,
            texture_scaler,
        ),
        "test": EyeFusionDataset(
            split_frames["test"],
            mode,
            eval_transform,
            texture_scaler,
        ),
    }

    generator = torch.Generator()
    generator.manual_seed(CONFIG["seed"])

    loaders = {
        "train": DataLoader(
            datasets["train"],
            batch_size=CONFIG["batch_size"],
            shuffle=True,
            num_workers=CONFIG["num_workers"],
            pin_memory=True,
            generator=generator,
        ),

        "val": DataLoader(
            datasets["val"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=True,
        ),

        "test": DataLoader(
            datasets["test"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=True,
        ),
    }

    return datasets, loaders


print("Dataset and augmentation pipeline ready.")

Dataset and augmentation pipeline ready.


In [37]:
# ============================================================
# 10. SWIN V2 + TEXTURE FUSION MODEL
# ============================================================
class SwinTextureFusion(nn.Module):
    def __init__(
        self,
        texture_dim: int,
        mode: str,
        hidden_dim: int,
        dropout: float,
        pretrained: bool = True,
    ):
        super().__init__()
        self.mode = mode
        self.use_swin = mode != "texture_only"
        self.use_texture = mode != "swin_only"

        self.swin_dim = 0
        if self.use_swin:
            selected_weights = Swin_V2_T_Weights.DEFAULT if pretrained else None
            self.backbone = swin_v2_t(weights=selected_weights)
            self.swin_dim = self.backbone.head.in_features
            self.backbone.head = nn.Identity()
            self.swin_projection = nn.Sequential(
                nn.LayerNorm(self.swin_dim),
                nn.Linear(self.swin_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        else:
            self.backbone = None
            self.swin_projection = None

        self.texture_dim = texture_dim
        if self.use_texture:
            self.texture_projection = nn.Sequential(
                nn.LayerNorm(texture_dim),
                nn.Linear(texture_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        else:
            self.texture_projection = None

        fusion_dim = hidden_dim * int(self.use_swin) + hidden_dim * int(
            self.use_texture
        )

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(
        self,
        image: torch.Tensor,
        texture: torch.Tensor,
    ) -> torch.Tensor:
        branches = []

        if self.use_swin:
            deep_features = self.backbone(image)
            branches.append(self.swin_projection(deep_features))

        if self.use_texture:
            branches.append(self.texture_projection(texture))

        fused = torch.cat(branches, dim=1)
        return self.classifier(fused).squeeze(1)

    def freeze_backbone(self) -> None:
        if self.backbone is not None:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

    def unfreeze_backbone_tail(self) -> None:
        if self.backbone is None:
            return

        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        # Unfreeze final feature stage, final norm and projection.
        for parameter in self.backbone.features[-1].parameters():
            parameter.requires_grad = True
        for parameter in self.backbone.norm.parameters():
            parameter.requires_grad = True
        for parameter in self.swin_projection.parameters():
            parameter.requires_grad = True

def texture_dimension_for_mode(mode: str) -> int:
    dummy = np.zeros(len(FEATURE_NAMES), dtype=np.float32)
    return len(select_texture_groups(dummy, mode))

print("Model definition ready.")

Model definition ready.


In [38]:
# ============================================================
# 11. METRICS, THRESHOLD, OPTIMIZER AND CHECKPOINT STATE
# ============================================================

def binary_metrics(
    targets: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> Dict[str, float]:

    predictions = (
        probabilities >= threshold
    ).astype(int)

    result = {
        "accuracy": accuracy_score(
            targets,
            predictions,
        ),

        "balanced_accuracy": balanced_accuracy_score(
            targets,
            predictions,
        ),

        "precision": precision_score(
            targets,
            predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            targets,
            predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            targets,
            predictions,
            zero_division=0,
        ),
    }

    if len(np.unique(targets)) == 2:
        result["roc_auc"] = roc_auc_score(
            targets,
            probabilities,
        )

        result["average_precision"] = (
            average_precision_score(
                targets,
                probabilities,
            )
        )

    else:
        result["roc_auc"] = float("nan")
        result["average_precision"] = float("nan")

    return {
        key: float(value)
        for key, value in result.items()
    }


def find_best_threshold(
    targets: np.ndarray,
    probabilities: np.ndarray,
) -> Tuple[float, float]:

    thresholds = np.linspace(
        CONFIG["threshold_search_min"],
        CONFIG["threshold_search_max"],
        CONFIG["threshold_search_steps"],
    )

    best_threshold = CONFIG["threshold"]
    best_f1 = -np.inf

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            targets,
            predictions,
            zero_division=0,
        )

        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_f1


def build_optimizer(
    model: SwinTextureFusion,
):

    backbone_parameters = []
    head_parameters = []

    for name, parameter in model.named_parameters():

        if not parameter.requires_grad:
            continue

        if name.startswith("backbone."):
            backbone_parameters.append(parameter)
        else:
            head_parameters.append(parameter)

    parameter_groups = []

    if backbone_parameters:
        parameter_groups.append({
            "params": backbone_parameters,
            "lr": CONFIG["backbone_learning_rate"],
        })

    if head_parameters:
        parameter_groups.append({
            "params": head_parameters,
            "lr": CONFIG["learning_rate"],
        })

    return torch.optim.AdamW(
        parameter_groups,
        weight_decay=CONFIG["weight_decay"],
    )


def capture_rng_state() -> Dict[str, Any]:

    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
    }

    if torch.cuda.is_available():
        state["cuda_rng_state"] = (
            torch.cuda.get_rng_state_all()
        )

    return state


def checkpoint_state(
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    best_metric: float,
    history: List[Dict[str, float]],
    mode: str,
    best_threshold: float = 0.50,
) -> Dict[str, Any]:

    state = {
        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "best_metric_score":
            float(best_metric),

        "best_threshold":
            float(best_threshold),

        "selection_metric":
            "val_roc_auc",

        "history":
            history,

        "config":
            CONFIG,

        "mode":
            mode,

        "feature_slices": {
            name: [
                value.start,
                value.stop,
            ]
            for name, value
            in FEATURE_SLICES.items()
        },
    }

    state.update(
        capture_rng_state()
    )

    return state


print("Metrics, threshold search and checkpoint helpers ready.")

Metrics, threshold search and checkpoint helpers ready.


In [39]:
# ============================================================
# 12. TRAIN AND EVALUATION LOOPS
# ============================================================
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer=None,
    scaler=None,
    training: bool = False,
    max_batches: Optional[int] = None,
) -> Tuple[float, np.ndarray, np.ndarray]:
    model.train(training)
    losses = []
    targets_all = []
    probabilities_all = []

    for batch_index, batch in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break

        images = batch["image"].to(DEVICE, non_blocking=True)
        textures = batch["texture"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            enabled=CONFIG["amp"] and DEVICE.type == "cuda",
        ):
            logits = model(images, textures)
            loss = criterion(logits, targets)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"NaN/Inf loss detected at batch {batch_index}"
            )

        if training:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CONFIG["gradient_clip_norm"],
            )
            scaler.step(optimizer)
            scaler.update()

        losses.append(float(loss.detach().cpu()))
        probabilities = torch.sigmoid(logits).detach().cpu().numpy()
        probabilities_all.extend(probabilities.tolist())
        targets_all.extend(targets.detach().cpu().numpy().tolist())

    return (
        float(np.mean(losses)),
        np.asarray(targets_all, dtype=np.int64),
        np.asarray(probabilities_all, dtype=np.float64),
    )

def predict_loader(
    model: nn.Module,
    loader: DataLoader,
) -> pd.DataFrame:
    model.eval()
    records = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            images = batch["image"].to(DEVICE, non_blocking=True)
            textures = batch["texture"].to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=CONFIG["amp"] and DEVICE.type == "cuda",
            ):
                logits = model(images, textures)

            probabilities = torch.sigmoid(logits).cpu().numpy()
            targets = batch["target"].numpy()

            for index in range(len(probabilities)):
                records.append(
                    {
                        "sample_id": batch["sample_id"][index],
                        "video_id": batch["video_id"][index],
                        "image_path": batch["image_path"][index],
                        "target": int(targets[index]),
                        "probability_fake": float(probabilities[index]),
                    }
                )

    predictions = pd.DataFrame(records)
    predictions["prediction"] = (
        predictions["probability_fake"] >= CONFIG["threshold"]
    ).astype(int)
    predictions["correct"] = (
        predictions["prediction"] == predictions["target"]
    )
    return predictions

In [40]:
# ============================================================
# 13. SMOKE TEST
# Full training is blocked if this cell fails.
# ============================================================
def smoke_test(mode: str) -> None:
    datasets, loaders = build_loaders(mode)
    texture_dim = texture_dimension_for_mode(mode)

    model = SwinTextureFusion(
        texture_dim=texture_dim,
        mode=mode,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
        pretrained=CONFIG["pretrained"],
    ).to(DEVICE)

    if model.use_swin:
        model.freeze_backbone()

    optimizer = build_optimizer(model)

    train_targets = datasets["train"].df["target"].to_numpy()
    classes = np.array([0, 1])
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=train_targets,
    )
    positive_weight = torch.tensor(
        [class_weights[1] / class_weights[0]],
        device=DEVICE,
        dtype=torch.float32,
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    grad_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=CONFIG["amp"] and DEVICE.type == "cuda",
    )

    train_loss, _, _ = run_epoch(
        model,
        loaders["train"],
        criterion,
        optimizer=optimizer,
        scaler=grad_scaler,
        training=True,
        max_batches=CONFIG["smoke_train_batches"],
    )
    val_loss, _, _ = run_epoch(
        model,
        loaders["val"],
        criterion,
        training=False,
        max_batches=CONFIG["smoke_val_batches"],
    )

    test_checkpoint = DIRS["checkpoints"] / "smoke_test.ckpt"
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, CONFIG["epochs"]),
    )
    state = checkpoint_state(
        epoch=0,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=grad_scaler,
        best_metric=0.0,
        history=[],
        mode=mode,
    )
    atomic_torch_save(state, test_checkpoint)

    reloaded = torch.load(
        test_checkpoint,
        map_location="cpu",
        weights_only=False,
    )
    assert "model_state_dict" in reloaded
    test_checkpoint.unlink(missing_ok=True)

    del model, optimizer, loaders, datasets
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        f"Smoke test passed | mode={mode} | "
        f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f}"
    )

if CONFIG["smoke_test"]:
    smoke_test(CONFIG["active_mode"])
else:
    print("Smoke test disabled in configuration.")

Smoke test passed | mode=full | train_loss=0.69192 | val_loss=0.69286


In [41]:
# ============================================================
# 14. FULL TRAINING FUNCTION
# Best checkpoint: Validation ROC-AUC
# Threshold: Best validation F1 threshold
# ============================================================

def train_experiment(
    mode: str,
) -> Dict[str, Any]:

    print(
        f"\n{'=' * 70}\n"
        f"Training mode: {mode}\n"
        f"{'=' * 70}"
    )

    mode_dir = RUN_DIR / mode

    mode_dirs = {
        "root":
            mode_dir,

        "checkpoints":
            mode_dir / "checkpoints",

        "metrics":
            mode_dir / "metrics",

        "predictions":
            mode_dir / "predictions",

        "figures":
            mode_dir / "figures",
    }

    for path in mode_dirs.values():
        path.mkdir(
            parents=True,
            exist_ok=True,
        )

    datasets, loaders = build_loaders(mode)

    texture_dim = texture_dimension_for_mode(mode)

    model = SwinTextureFusion(
        texture_dim=texture_dim,
        mode=mode,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
        pretrained=CONFIG["pretrained"],
    ).to(DEVICE)

    if (
        model.use_swin
        and CONFIG["freeze_backbone_epochs"] > 0
    ):
        model.freeze_backbone()

    train_targets = (
        datasets["train"]
        .df["target"]
        .to_numpy()
    )

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_targets,
    )

    positive_weight = torch.tensor(
        [
            class_weights[1]
            / class_weights[0]
        ],
        device=DEVICE,
        dtype=torch.float32,
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=positive_weight
    )

    optimizer = build_optimizer(model)

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(
                1,
                CONFIG["epochs"],
            ),
        )
    )

    grad_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            CONFIG["amp"]
            and DEVICE.type == "cuda"
        ),
    )

    history = []

    best_auc = -np.inf
    best_threshold = CONFIG["threshold"]

    patience_counter = 0

    for epoch in range(
        CONFIG["epochs"]
    ):

        if (
            model.use_swin
            and CONFIG["unfreeze_last_stages"]
            and epoch
            == CONFIG["freeze_backbone_epochs"]
        ):

            model.unfreeze_backbone_tail()

            optimizer = build_optimizer(model)

            scheduler = (
                torch.optim.lr_scheduler
                .CosineAnnealingLR(
                    optimizer,
                    T_max=max(
                        1,
                        CONFIG["epochs"] - epoch,
                    ),
                )
            )

            print(
                "Final Swin stage unfrozen."
            )

        train_loss, train_y, train_p = run_epoch(
            model,
            loaders["train"],
            criterion,
            optimizer=optimizer,
            scaler=grad_scaler,
            training=True,
        )

        val_loss, val_y, val_p = run_epoch(
            model,
            loaders["val"],
            criterion,
            training=False,
        )

        train_metrics = binary_metrics(
            train_y,
            train_p,
            CONFIG["threshold"],
        )

        epoch_threshold, epoch_threshold_f1 = (
            find_best_threshold(
                val_y,
                val_p,
            )
        )

        val_metrics = binary_metrics(
            val_y,
            val_p,
            epoch_threshold,
        )

        row = {
            "epoch":
                epoch + 1,

            "train_loss":
                train_loss,

            "val_loss":
                val_loss,

            "validation_threshold":
                epoch_threshold,

            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },

            **{
                f"val_{key}": value
                for key, value
                in val_metrics.items()
            },

            "learning_rate":
                optimizer.param_groups[0]["lr"],
        }

        history.append(row)

        atomic_write_csv(
            pd.DataFrame(history),
            mode_dirs["metrics"]
            / "epoch_metrics.csv",
        )

        current_auc = val_metrics["roc_auc"]

        last_state = checkpoint_state(
            epoch=epoch + 1,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=grad_scaler,
            best_metric=best_auc,
            history=history,
            mode=mode,
            best_threshold=best_threshold,
        )

        atomic_torch_save(
            last_state,
            mode_dirs["checkpoints"]
            / "last.ckpt",
        )

        if (
            np.isfinite(current_auc)
            and current_auc > best_auc
        ):

            best_auc = current_auc
            best_threshold = epoch_threshold
            patience_counter = 0

            best_state = checkpoint_state(
                epoch=epoch + 1,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=grad_scaler,
                best_metric=best_auc,
                history=history,
                mode=mode,
                best_threshold=best_threshold,
            )

            atomic_torch_save(
                best_state,
                mode_dirs["checkpoints"]
                / "best.ckpt",
            )

        else:
            patience_counter += 1

        scheduler.step()

        print(
            f"Epoch {epoch + 1:02d}/"
            f"{CONFIG['epochs']} | "

            f"Train Loss "
            f"{train_loss:.4f} | "

            f"Val Loss "
            f"{val_loss:.4f} | "

            f"Val F1 "
            f"{val_metrics['f1']:.4f} | "

            f"Val AUC "
            f"{val_metrics['roc_auc']:.4f} | "

            f"Threshold "
            f"{epoch_threshold:.3f}"
        )

        if (
            patience_counter
            >= CONFIG["patience"]
        ):
            print(
                "Early stopping activated."
            )
            break

    best_checkpoint = torch.load(
        mode_dirs["checkpoints"]
        / "best.ckpt",
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    selected_threshold = float(
        best_checkpoint.get(
            "best_threshold",
            CONFIG["threshold"],
        )
    )

    print(
        f"Best epoch: "
        f"{best_checkpoint['epoch']}"
    )

    print(
        f"Best validation ROC-AUC: "
        f"{best_checkpoint['best_metric_score']:.4f}"
    )

    print(
        f"Selected validation threshold: "
        f"{selected_threshold:.3f}"
    )

    test_predictions = predict_loader(
        model,
        loaders["test"],
    )

    test_predictions["prediction"] = (
        test_predictions[
            "probability_fake"
        ] >= selected_threshold
    ).astype(int)

    test_predictions["correct"] = (
        test_predictions["prediction"]
        == test_predictions["target"]
    )

    atomic_write_csv(
        test_predictions,
        mode_dirs["predictions"]
        / "test_predictions.csv",
    )

    test_metrics = binary_metrics(
        test_predictions[
            "target"
        ].to_numpy(),

        test_predictions[
            "probability_fake"
        ].to_numpy(),

        selected_threshold,
    )

    test_metrics.update({
        "mode":
            mode,

        "best_epoch":
            int(
                best_checkpoint["epoch"]
            ),

        "selection_metric":
            "val_roc_auc",

        "best_validation_auc":
            float(
                best_checkpoint[
                    "best_metric_score"
                ]
            ),

        "selected_threshold":
            selected_threshold,

        "texture_dimension":
            int(texture_dim),

        "sample_count":
            int(
                len(test_predictions)
            ),
    })

    atomic_write_json(
        test_metrics,
        mode_dirs["metrics"]
        / "test_metrics.json",
    )

    del model
    del optimizer
    del loaders
    del datasets

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "mode":
            mode,

        **test_metrics,

        "mode_dir":
            str(mode_dir),
    }

In [42]:
# ============================================================
# 15. RUN MAIN EXPERIMENT OR ALL ABLATIONS
# ============================================================
if CONFIG["run_all_ablations"]:
    modes_to_run = CONFIG["ablation_modes"]
else:
    modes_to_run = [CONFIG["active_mode"]]

experiment_results = []
for mode in modes_to_run:
    experiment_results.append(train_experiment(mode))

results_df = pd.DataFrame(experiment_results)
atomic_write_csv(results_df, DIRS["metrics"] / "ablation_results.csv")
display(results_df)


Training mode: full
Epoch 01/30 | Train Loss 0.6650 | Val Loss 0.6340 | Val F1 0.6844 | Val AUC 0.6907 | Threshold 0.395
Epoch 02/30 | Train Loss 0.6377 | Val Loss 0.6390 | Val F1 0.6943 | Val AUC 0.6952 | Threshold 0.360
Epoch 03/30 | Train Loss 0.6272 | Val Loss 0.6272 | Val F1 0.6959 | Val AUC 0.6971 | Threshold 0.380
Epoch 04/30 | Train Loss 0.6147 | Val Loss 0.6229 | Val F1 0.7059 | Val AUC 0.7011 | Threshold 0.305
Epoch 05/30 | Train Loss 0.6050 | Val Loss 0.6157 | Val F1 0.7115 | Val AUC 0.7222 | Threshold 0.265
Final Swin stage unfrozen.
Epoch 06/30 | Train Loss 0.6022 | Val Loss 0.6329 | Val F1 0.6828 | Val AUC 0.6889 | Threshold 0.405
Epoch 07/30 | Train Loss 0.6017 | Val Loss 0.6406 | Val F1 0.6737 | Val AUC 0.6809 | Threshold 0.390
Epoch 08/30 | Train Loss 0.5925 | Val Loss 0.6390 | Val F1 0.6761 | Val AUC 0.6812 | Threshold 0.410
Epoch 09/30 | Train Loss 0.5898 | Val Loss 0.6307 | Val F1 0.6759 | Val AUC 0.7038 | Threshold 0.360
Epoch 10/30 | Train Loss 0.5967 | Val Loss 

Predicting:   0%|          | 0/19 [00:00<?, ?it/s]

,mode,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,best_epoch,selection_metric,best_validation_auc,selected_threshold,texture_dimension,sample_count,mode_dir
0,full,0.612583,0.603925,0.584416,0.865385,0.697674,0.709036,0.733504,5,val_roc_auc,0.722237,0.265,240,302,/content/drive/MyDrive/AISC DeepFake Çalışmala...


In [43]:
# ============================================================
# 16. MAIN MODE FIGURES
# ============================================================
MAIN_MODE = CONFIG["active_mode"]
MODE_DIR = RUN_DIR / MAIN_MODE
history = pd.read_csv(MODE_DIR / "metrics" / "epoch_metrics.csv")
predictions = pd.read_csv(
    MODE_DIR / "predictions" / "test_predictions.csv"
)

def save_mode_figure(fig: plt.Figure, filename: str) -> None:
    target = MODE_DIR / "figures" / filename
    fig.savefig(target, dpi=150, bbox_inches="tight")
    plt.close(fig)
    with Image.open(target) as image:
        assert min(image.size) >= 600, (
            f"Figure resolution is insufficient: {image.size}"
        )

# Loss curve.
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(history["epoch"], history["train_loss"], label="Training Loss")
ax.plot(history["epoch"], history["val_loss"], label="Validation Loss")
ax.set_title("Training and Validation Loss Curve")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_mode_figure(fig, "01_training_validation_loss_curve.png")

# F1 curve.
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(history["epoch"], history["train_f1"], label="Training F1")
ax.plot(history["epoch"], history["val_f1"], label="Validation F1")
ax.set_title("Training and Validation F1 Curve")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1 Score")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_mode_figure(fig, "02_training_validation_f1_curve.png")

# Confusion matrix.
cm = confusion_matrix(
    predictions["target"],
    predictions["prediction"],
    labels=[0, 1],
)
fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
image = ax.imshow(cm)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])
ax.set_title("Test Confusion Matrix")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
fig.colorbar(image, ax=ax)
fig.tight_layout()
save_mode_figure(fig, "03_test_confusion_matrix.png")

# ROC curve.
fpr, tpr, _ = roc_curve(
    predictions["target"],
    predictions["probability_fake"],
)
auc_value = roc_auc_score(
    predictions["target"],
    predictions["probability_fake"],
)
fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
ax.plot(fpr, tpr, label=f"ROC AUC = {auc_value:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
ax.set_title("Test Receiver Operating Characteristic Curve")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_mode_figure(fig, "04_test_roc_curve.png")

# Precision-recall curve.
precision, recall, _ = precision_recall_curve(
    predictions["target"],
    predictions["probability_fake"],
)
ap_value = average_precision_score(
    predictions["target"],
    predictions["probability_fake"],
)
fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
ax.plot(recall, precision, label=f"Average Precision = {ap_value:.4f}")
ax.set_title("Test Precision-Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_mode_figure(fig, "05_test_precision_recall_curve.png")

print("Main experiment figures saved.")

Main experiment figures saved.


In [44]:
# ============================================================
# 17. VIDEO-LEVEL EVALUATION
# ============================================================
video_predictions = (
    predictions.groupby("video_id", as_index=False)
    .agg(
        target=("target", "first"),
        probability_fake=("probability_fake", "mean"),
        sample_count=("sample_id", "count"),
    )
)
video_predictions["prediction"] = (
    video_predictions["probability_fake"] >= CONFIG["threshold"]
).astype(int)
video_predictions["correct"] = (
    video_predictions["prediction"] == video_predictions["target"]
)

atomic_write_csv(
    video_predictions,
    MODE_DIR / "predictions" / "video_level_predictions.csv",
)

video_metrics = binary_metrics(
    video_predictions["target"].to_numpy(),
    video_predictions["probability_fake"].to_numpy(),
    CONFIG["threshold"],
)
atomic_write_json(
    video_metrics,
    MODE_DIR / "metrics" / "video_level_metrics.json",
)

print(json.dumps(video_metrics, indent=2))

{
  "accuracy": 1.0,
  "balanced_accuracy": 1.0,
  "precision": 1.0,
  "recall": 1.0,
  "f1": 1.0,
  "roc_auc": 1.0,
  "average_precision": 1.0
}


In [45]:
# ============================================================
# 18. ABLATION COMPARISON FIGURE
# Run after enabling run_all_ablations=True.
# ============================================================
ablation_path = DIRS["metrics"] / "ablation_results.csv"
ablation = pd.read_csv(ablation_path)

if len(ablation) > 1:
    fig, ax = plt.subplots(figsize=(12, 7), dpi=150)
    x = np.arange(len(ablation))
    width = 0.25

    ax.bar(x - width, ablation["accuracy"], width, label="Accuracy")
    ax.bar(x, ablation["f1"], width, label="F1 Score")
    ax.bar(x + width, ablation["roc_auc"], width, label="ROC AUC")

    ax.set_xticks(x, ablation["mode"], rotation=20, ha="right")
    ax.set_title("Ablation Study Performance Comparison")
    ax.set_xlabel("Feature Configuration")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    save_figure(fig, "02_ablation_performance_comparison.png")
else:
    print(
        "Only one mode was trained. Set run_all_ablations=True "
        "to generate the ablation comparison."
    )

Only one mode was trained. Set run_all_ablations=True to generate the ablation comparison.


In [46]:
# ============================================================
# 19. FINAL QUALITY GATES
# ============================================================
def validate_run(mode: str) -> Dict[str, Any]:
    mode_dir = RUN_DIR / mode
    required_files = [
        mode_dir / "checkpoints" / "last.ckpt",
        mode_dir / "checkpoints" / "best.ckpt",
        mode_dir / "metrics" / "epoch_metrics.csv",
        mode_dir / "metrics" / "test_metrics.json",
        mode_dir / "predictions" / "test_predictions.csv",
    ]
    missing = [str(path) for path in required_files if not path.exists()]
    if missing:
        raise AssertionError(f"Required experiment outputs are missing: {missing}")

    checkpoint = torch.load(
        mode_dir / "checkpoints" / "best.ckpt",
        map_location="cpu",
        weights_only=False,
    )
    assert "model_state_dict" in checkpoint
    assert "optimizer_state_dict" in checkpoint
    assert "scheduler_state_dict" in checkpoint
    assert "scaler_state_dict" in checkpoint

    epoch_metrics = pd.read_csv(
        mode_dir / "metrics" / "epoch_metrics.csv"
    )
    numeric_columns = epoch_metrics.select_dtypes(include=[np.number]).columns
    if not np.isfinite(epoch_metrics[numeric_columns].to_numpy()).all():
        raise FloatingPointError("NaN/Inf detected in epoch metrics.")

    test_predictions = pd.read_csv(
        mode_dir / "predictions" / "test_predictions.csv"
    )
    assert len(test_predictions) > 0
    assert test_predictions["sample_id"].is_unique

    return {
        "mode": mode,
        "required_outputs": "PASSED",
        "checkpoint_integrity": "PASSED",
        "numeric_metrics": "PASSED",
        "prediction_uniqueness": "PASSED",
    }

quality_results = [validate_run(mode) for mode in modes_to_run]
atomic_write_json(
    {"quality_gates": quality_results},
    DIRS["audit"] / "final_quality_gates.json",
)

print(json.dumps(quality_results, indent=2))
print("\nExperiment completed successfully.")
print("Output directory:", RUN_DIR)

[
  {
    "mode": "full",
    "required_outputs": "PASSED",
    "checkpoint_integrity": "PASSED",
    "numeric_metrics": "PASSED",
    "prediction_uniqueness": "PASSED"
  }
]

Experiment completed successfully.
Output directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Sonuçlar/SwinV2_TextureFusion/20260806_1748_eye_swinv2_texturefusion_seed42
